<a href="https://colab.research.google.com/github/cantika-alff/2025_PBO_TI1B/blob/main/Jobsheet_AI10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack

In [9]:
# Load dataset
df = pd.read_csv("spotify-2023.csv", encoding='latin1')
df

,track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,...,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
0,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703,43,...,125,B,Major,80,89,83,31,0,8,4
1,LALA,Myke Towers,1,2023,3,23,1474,48,133716286,48,...,92,C#,Major,71,61,74,7,0,10,4
2,vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974,94,...,138,F,Major,51,32,53,17,0,31,6
3,Cruel Summer,Taylor Swift,1,2019,8,23,7858,100,800840817,116,...,170,A,Major,55,58,72,11,0,11,15
4,WHERE SHE GOES,Bad Bunny,1,2023,5,18,3133,50,303236322,84,...,144,A,Minor,65,23,80,14,63,11,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
948,My Mind & Me,Selena Gomez,1,2022,11,3,953,0,91473363,61,...,144,A,Major,60,24,39,57,0,8,3
949,Bigger Than The Whole Sky,Taylor Swift,1,2022,10,21,1180,0,121871870,4,...,166,F#,Major,42,7,24,83,1,12,6
950,A Veces (feat. Feid),"Feid, Paulo Londra",2,2022,11,3,573,0,73513683,2,...,92,C#,Major,80,81,67,4,0,8,6
951,En La De Ella,"Feid, Sech, Jhayco",3,2022,10,20,1320,0,133895612,29,...,97,C#,Major,82,67,77,8,0,12,5


In [10]:
# Preprocessing
for col in ['streams', 'in_deezer_playlists', 'in_shazam_charts']:
    df[col] = df[col].astype(str).str.replace(",", "")
    df[col] = pd.to_numeric(df[col], errors='coerce')

for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

for col in df.select_dtypes(include='number').columns:
    df[col].fillna(df[col].median(), inplace=True)

categorical_features = ['artist(s)_name', 'key', 'mode']
numerical_features = [
    'bpm', 'danceability_%', 'valence_%', 'energy_%', 'acousticness_%',
    'instrumentalness_%', 'liveness_%', 'speechiness_%'
]

df['text_features'] = df[categorical_features].agg(' '.join, axis=1)

scaler = MinMaxScaler()
numerical_scaled = scaler.fit_transform(df[numerical_features])
numerical_scaled_df = pd.DataFrame(numerical_scaled, columns=numerical_features)

full_features = pd.concat([df[['track_name', 'text_features']].reset_index(drop=True), numerical_scaled_df], axis=1)

/tmp/ipython-input-10-1477246204.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)
/tmp/ipython-input-10-1477246204.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', 

In [11]:
# TF-IDF vectorization
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(full_features['text_features'])

# Combine TF-IDF and numeric features
final_matrix = hstack([tfidf_matrix, numerical_scaled_df.values])

# Compute similarity
similarity = cosine_similarity(final_matrix)

In [16]:
def recommend(track_name, top_n=10):
    if track_name not in full_features['track_name'].values:
        return f"Lagu '{track_name}' tidak ditemukan dalam dataset."
    idx = full_features[full_features['track_name'] == track_name].index[0]
    sim_scores = list(enumerate(similarity[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    top_indices = [i for i, score in sim_scores[1:top_n+1]]
    return df.iloc[top_indices][['track_name', 'artist(s)_name']]

In [17]:
# Contoh penggunaan
if __name__ == "__main__":
    print("Sistem Rekomendasi Lagu - Content Based")
    user_input = input("Masukkan judul lagu: ")
    results = recommend(user_input)
    print("\nRekomendasi:")
    print(results)


Sistem Rekomendasi Lagu - Content Based
Masukkan judul lagu: vampire

Rekomendasi:
                     track_name                     artist(s)_name
565                     deja vu                     Olivia Rodrigo
513                    good 4 u                     Olivia Rodrigo
721          jealousy, jealousy                     Olivia Rodrigo
535             drivers license                     Olivia Rodrigo
588                     happier                     Olivia Rodrigo
596              favorite crime                     Olivia Rodrigo
367       Bombonzinho - Ao Vivo     Israel & Rodolffo, Ana Castela
229  Seu Brilho Sumiu - Ao Vivo  Israel & Rodolffo, Mari Fernandez
240      Erro Gostoso - Ao Vivo                      Simone Mendes
409    Eu Gosto Assim - Ao Vivo      Gustavo Mioto, Mari Fernandez


In [28]:
def recommend(track_name, top_n=10):
    track_name_lower = track_name.lower()
    track_names_lower = df_clean["track_name"].str.lower()

    if track_name_lower not in track_names_lower.values:
        return f"Lagu '{track_name}' tidak ditemukan dalam dataset."

    try:
        idx = track_names_lower[track_names_lower == track_name_lower].index[0]
        sim_scores = list(enumerate(similarity_matrix[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        top_indices = sim_scores[1:top_n+1]

        results = []
        for i, score in top_indices:
            results.append({
                "track_name": df_clean.iloc[i]["track_name"],
                "artist": df_clean.iloc[i]["artist(s)_name"],
                "similarity": round(score, 4)
            })

        return results  # <== PASTIKAN INI ADA
    except Exception as e:
        return f"Terjadi kesalahan: {str(e)}"


In [30]:
if __name__ == "__main__":
    print("Sistem Rekomendasi Lagu - Content Based")
    user_input = input("Masukkan judul lagu: ")
    results = recommend(user_input)

    print("\nRekomendasi:")
    if isinstance(results, str):
        print(results)
    else:
        for i, r in enumerate(results, start=1):
            print(f"{i}. {r['track_name']} - {r['artist']} (Similarity: {r['similarity']})")


Sistem Rekomendasi Lagu - Content Based
Masukkan judul lagu: LAla

Rekomendasi:
1. Besos Moja2 - Wisin & Yandel, ROSALï¿½ (Similarity: 0.9825)
2. Adore You - Harry Styles (Similarity: 0.9655)
3. SPIT IN MY FACE! - ThxSoMch (Similarity: 0.9634)
4. Wait a Minute! - Willow (Similarity: 0.9533)
5. Blank Space - Taylor Swift (Similarity: 0.9511)
6. One Right Now (with The Weeknd) - The Weeknd, Post Malone (Similarity: 0.9487)
7. La Santa - Daddy Yankee, Bad Bunny (Similarity: 0.9373)
8. X ï¿½ï¿½LTIMA - Daddy Yankee, Bad Bunny (Similarity: 0.917)
9. SloMo - Chanel (Similarity: 0.9138)
10. En La De Ella - Feid, Sech, Jhayco (Similarity: 0.9035)


ihiohp

In [31]:
def recommend(track_name, top_n=10):
    if track_name not in full_features['track_name'].values:
        return f"Lagu '{track_name}' tidak ditemukan dalam dataset."

    idx = full_features[full_features['track_name'] == track_name].index[0]
    sim_scores = list(enumerate(similarity[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    top_indices = [i for i, score in sim_scores[1:top_n+1]]

    return df.iloc[top_indices][['track_name', 'artist(s)_name']]


In [32]:
# Contoh penggunaan
if __name__ == "__main__":
    print("Sistem Rekomendasi Lagu - Content Based")
    user_input = input("Masukkan judul lagu: ")
    results = recommend(user_input)
    print("\nRekomendasi:")
    print(results)


Sistem Rekomendasi Lagu - Content Based
Masukkan judul lagu: vampire

Rekomendasi:
                     track_name                     artist(s)_name
565                     deja vu                     Olivia Rodrigo
513                    good 4 u                     Olivia Rodrigo
721          jealousy, jealousy                     Olivia Rodrigo
535             drivers license                     Olivia Rodrigo
588                     happier                     Olivia Rodrigo
596              favorite crime                     Olivia Rodrigo
367       Bombonzinho - Ao Vivo     Israel & Rodolffo, Ana Castela
229  Seu Brilho Sumiu - Ao Vivo  Israel & Rodolffo, Mari Fernandez
240      Erro Gostoso - Ao Vivo                      Simone Mendes
409    Eu Gosto Assim - Ao Vivo      Gustavo Mioto, Mari Fernandez


In [33]:
def recommend(track_name, top_n=10):
    track_name_lower = track_name.lower()
    track_names_lower = full_features['track_name'].str.lower()

    if track_name_lower not in track_names_lower.values:
        return f"Lagu '{track_name}' tidak ditemukan dalam dataset."

    idx = track_names_lower[track_names_lower == track_name_lower].index[0]
    sim_scores = list(enumerate(similarity[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    top_indices = [i for i, score in sim_scores[1:top_n+1]]

    return df.iloc[top_indices][['track_name', 'artist(s)_name']]


In [34]:
# Contoh penggunaan
if __name__ == "__main__":
    print("Sistem Rekomendasi Lagu - Content Based")
    user_input = input("Masukkan judul lagu: ")
    results = recommend(user_input)
    print("\nRekomendasi:")
    print(results)


Sistem Rekomendasi Lagu - Content Based
Masukkan judul lagu: LAla

Rekomendasi:
                                track_name                      artist(s)_name
346                      PLAYA DEL INGLï¿½                Myke Towers, Quevedo
647                                Sï¿½ï¿½  Anuel Aa, Myke Towers, Jhay Cortez
50                                El Cielo    Feid, Myke Towers, Sky Rompiendo
795  That That (prod. & feat. SUGA of BTS)                           PSY, Suga
602                              The Feels                               TWICE
423                      Super Freaky Girl                         Nicki Minaj
676                            A Tu Merced                           Bad Bunny
586                             DANCE CRIP                              Trueno
754        There's Nothing Holdin' Me Back                        Shawn Mendes
144                                  QUEMA        Sog, Ryan Castro, Peso Pluma


In [35]:
def recommend(track_name, top_n=5):
    if track_name not in full_features['track_name'].values:
        return f"Lagu '{track_name}' tidak ditemukan dalam dataset."

    idx = full_features[full_features['track_name'] == track_name].index[0]
    sim_scores = list(enumerate(similarity[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    top_scores = sim_scores[1:top_n+1]

    results = []
    for i, score in top_scores:
        results.append({
            'track_name': df.iloc[i]['track_name'],
            'artist(s)_name': df.iloc[i]['artist(s)_name'],
            'similarity_score': round(score, 4)  # 4 angka di belakang koma
        })

    return pd.DataFrame(results)


In [37]:
# Contoh penggunaan
if __name__ == "__main__":
    print("Sistem Rekomendasi Lagu - Content Based")
    user_input = input("Masukkan judul lagu: ")
    results = recommend(user_input)
    print("\nRekomendasi:")
    print(results)


Sistem Rekomendasi Lagu - Content Based
Masukkan judul lagu: vampire

Rekomendasi:
           track_name  artist(s)_name  similarity_score
0             deja vu  Olivia Rodrigo            0.9412
1            good 4 u  Olivia Rodrigo            0.9321
2  jealousy, jealousy  Olivia Rodrigo            0.9240
3     drivers license  Olivia Rodrigo            0.8973
4             happier  Olivia Rodrigo            0.8795
